# bRAG: Basic (naive) RAG Implementation

This notebook demonstrates a complete implementation of a basic RAG system that enables question-answering over PDF documents. 
The implementation is model, database, and document loader agnostic, though it's currently configured with:
- LLM: DeepSeek (OpenAI-compatible API)
- Vector Database: Milvus (本地部署)
- Document Loader: PyPDFLoader

The system combines several key components:
1. Document Loading: Loads PDF documents (extensible to other document types)
2. Text Processing: Splits documents into manageable chunks
3. Vector Operations:
   - Embeds text using HuggingFace sentence-transformers model
   - Stores vectors in local Milvus vector database
4. Retrieval System: Implements efficient document retrieval
5. LLM Integration: Uses DeepSeek model for generating responses

All components can be swapped out for alternatives (e.g., different LLMs, vector stores, or document loaders) 
while maintaining the same overall architecture.

This implementation serves as a foundation for building more complex RAG applications
and can be customized based on specific use cases.

----------------------------------------

## Pre-requisites (optional but recommended)

### Only do the first step if you have never created a virtual environment for this repository. Otherwise, make sure that the Python Kernel that you selected is from your `venv/` folder.

In [82]:
# Create virtual environment
! python -m venv venv

Error: [Errno 13] Permission denied: 'e:\\own study code\\agent-llm\\bRAG-langchain-wdl\\venv\\Scripts\\python.exe'


In [83]:
# Activate virtual Python environment
! source venv/bin/activate

'source' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [84]:
# If your Python is not from your venv path, ensure that your IDE's kernel selection (on the top right corner) is set to the correct path 
# (your path output should contain "...venv/bin/python")

! which python

'which' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [85]:
# Install all packages
# ! pip install -r requirements.txt --quiet

## Environment

`(1) Packages`

In [ ]:
import os
# python-dotenv 是一个常用的工具，它允许你从 .env 文件中读取环境变量，这对于管理配置信息（如 API 密钥、数据库连接字符串等）非常有用，特别是当你不想将这些敏感信息直接硬编码在代码中时。
# from dotenv import load_dotenv

# Load all environment variables from .env file
# load_dotenv()

# Access the environment variables
langchain_tracing_v2 = os.getenv('LANGCHAIN_TRACING_V2')
langchain_endpoint = os.getenv('LANGCHAIN_ENDPOINT')
langchain_api_key = os.getenv('LANGCHAIN_API_KEY')

## LLM - DeepSeek (OpenAI-compatible)
api_key = os.getenv('DEEPSEEK_API_KEY').strip()
base_url = os.getenv('DEEPSEEK_API_BASE').strip()

print('api_key----',api_key)
print('base_url----',base_url)

## Milvus 本地向量数据库配置
milvus_host = os.getenv('MILVUS_HOST', 'localhost')  # 默认 localhost
milvus_port = os.getenv('MILVUS_PORT', '19530')       # 默认端口 19530
milvus_collection = os.getenv('MILVUS_COLLECTION', 'langchain_rag')  # collection 名称


`(2) LangSmith`

https://docs.smith.langchain.com/

In [87]:
os.environ['LANGCHAIN_TRACING_V2'] = langchain_tracing_v2
os.environ['LANGCHAIN_ENDPOINT'] = langchain_endpoint
os.environ['LANGCHAIN_API_KEY'] = langchain_api_key

`(3) API Keys`

In [88]:
# DeepSeek 使用 OpenAI 兼容接口，通过 base_url 指向 DeepSeek 服务
os.environ['OPENAI_API_KEY'] = api_key
os.environ['OPENAI_API_BASE'] = base_url
deepseek_model = "deepseek-chat"

`(4) Milvus 连接测试`

In [89]:
# 测试 Milvus Docker 服务是否正常运行
import socket

def check_milvus_connection(host='localhost', port=19530, timeout=3):
    try:
        sock = socket.create_connection((host, port), timeout=timeout)
        sock.close()
        print(f"✅ Milvus 服务正常：{host}:{port} 可以连接")
        return True
    except (socket.timeout, ConnectionRefusedError, OSError) as e:
        print(f"❌ Milvus 服务不可用：{host}:{port} - {e}")
        print("请确认 Docker 容器已启动：docker ps | grep milvus")
        return False

check_milvus_connection()

from pymilvus import connections

# 显式建立连接 (关键步骤！)
try:
    connections.connect(alias="default", host="localhost", port="19530")
    print("成功连接到 Milvus")
except Exception as e:
    print(f"连接 Milvus 失败: {e}")
    raise e


✅ Milvus 服务正常：localhost:19530 可以连接
成功连接到 Milvus


## Full RAG App (Basic)

In [ ]:
# 这段代码实现了一个 RAG（检索增强生成） 流程，用于从 PDF 文档中提取信息并基于 LLM（此处使用 DeepSeek 模型）回答问题。
from langchain_community.document_loaders import PyPDFLoader          # 用于加载PDF文档
from langchain_text_splitters import RecursiveCharacterTextSplitter     # 用于递归分割文本
from langchain_core.vectorstores import InMemoryVectorStore
# Milvus 是一个开源的高性能向量数据库，支持本地部署。
# langchain_milvus 提供了 LangChain 与 Milvus 的集成，操作方式与 Pinecone 类似。
from langchain_milvus import Milvus                                    # 用于 Milvus 向量存储
from langchain_core.output_parsers import StrOutputParser              # 用于解析输出为字符串
from langchain_core.runnables import RunnablePassthrough               # 用于传递运行时数据
from langchain_openai import ChatOpenAI                                # 用于 OpenAI 兼容聊天模型
# HuggingFaceEmbeddings：包装器，让你能在 LangChain 框架内，便捷地使用 Hugging Face 上成千上万的本地文本嵌入模型
# from langchain_huggingface import HuggingFaceEmbeddings                # 用于 HuggingFace 嵌入模型
# FastEmbedEmbeddings:包装器,
# FastEmbedEmbeddings 用的是 ONNX Runtime，完全不依赖 torch。
# from langchain_community.embeddings import FastEmbedEmbeddings
# from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import OllamaEmbeddings

from langchain_core.prompts import ChatPromptTemplate                       # 用于聊天提示模板

#### 1. 索引部分，用于文档处理和索引创建
# pdf_file_path = "test/langchain_turing.pdf"   # 定义PDF文件路径
pdf_file_path = "test/2026年周会任务.pdf"   # 定义PDF文件路径
loader = PyPDFLoader(pdf_file_path)            # 创建PDF加载器实例
# 语法：创建 PyPDFLoader 实例，调用 load() 方法返回文档列表（每个元素是一个 Document 对象，包含 page_content 和 metadata）
docs = loader.load()                           # 加载PDF文档内容

#### 2. Split 分割文本
# 作用：将长文档按 1000 字符分块，块间重叠 200 字符，保持语义连贯。
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
# 语法：split_documents() 接收文档列表，返回分割后的文档块列表。
splits = text_splitter.split_documents(docs)   # 使用文本分割器将文档分割成更小的块

print(f"文档块数量--0--: {len(splits)}")

#### 3. Embedding 模型
# 使用 HuggingFace 本地 embedding 模型，无需 API key，该模型将文本转换为向量

# 方案1：使用HuggingFaceEmbeddings
# embedding_model = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2",  # 指定 Hugging Face 模型库中的模型 ID 或本地路径；all-MiniLM-L6-v2：向量维度 384，速度快、占用资源少，性能均衡，是大多数任务的绝佳起点。
#     # model_kwargs={'device': 'cuda'}, # 强制使用 GPU 加速；或者 'device': 'cpu' 强制使用 CPU
#     # encode_kwargs={"normalize_embeddings": True}, # 对生成的向量进行 L2 归一化，对余弦相似度任务很重要
# )
# 放弃原因：sentence-transformers依赖 PyTorch，因网络问题，访问download.pytorch.org 超时；

# 方案2：使用FastEmbedEmbeddings
# embedding_model = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")
# 放弃原因： onnxruntime 的 DLL 也失败了，说明这个 venv 环境本身有问题，不只是 torch，整个 DLL 加载都不正常。

# 方案3：使用OpenAIEmbeddings
# embedding_model = OpenAIEmbeddings(
#     base_url="https://api.deepseek.com/v1",  # 1. 必须加上 /v1
#     api_key=api_key,                         # 2. 确保填入有效的 API Key
#     model="deepseek-chat"                    # 3. 使用 DeepSeek 支持的模型
# )
# 说明：由于没有OpenAI的API Key，所以选择使用兼容OpenAI的deepseek模型
# 放弃原因：deepseek没有对应的Embeddings模型，不支持将文本转换为向量，放弃该方案

# 方案4：使用Ollama部署本地模型（推荐，免费且无需 API Key）
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text" # 大小：274M
)

print(f"Embedding model--1--: {embedding_model}")
emb = embedding_model.embed_query("test")
print('emb----',len(emb))  # 应该是 768（nomic-embed-text 的维度）

#### 4. 创建 Milvus 向量存储（Milvus Lite 模式，无需 Docker）
# URI 使用本地 .db 文件路径，Milvus Lite 会自动创建该文件，无需手动建立
# 这是 Milvus 的嵌入式模式，适合本地开发和测试，无需启动任何服务
# URI = "./milvus.db"

# 第一步：初始化 Milvus vector store（连接数据库，指定 embedding 模型和索引参数）
# 注意：
# 1.使用本地文件数据库Milvus，需要安装milvus-lite
# 需要安装milvus-lite原因: Milvus Lite 是 Milvus 的嵌入式版本，可以在本地文件系统上运行，无需依赖外部服务。它适用于开发和测试环境，可以快速搭建起向量数据库，方便进行实验和原型开发。
# 安装方法：pip install milvus-lite 或 pip install pymilvus[milvus_lite] 一直提示下载失败
# 安装失败原因：milvus-lite 目前没有提供 Windows 平台 (.whl 文件) 的安装包;详见官网:https://pypi.org/project/milvus-lite/#files
# 使用方法:
# vector_store = Milvus(
#     embedding_function=embedding_model,        # 用于生成向量嵌入的模型
#     connection_args={"uri": URI},              # 使用本地文件作为数据库，自动创建
#     collection_name=milvus_collection,         # Milvus collection 名称 （langchain官方无）
#     index_params={"index_type": "FLAT", "metric_type": "L2"},  # 索引类型和距离度量
#     drop_old=True,                             # 每次运行时清空旧数据，重新写入
# )

# 2.使用远程服务器模式,使用 Docker 来运行 Milvus 服务器。
# 创建向量存储    
# 直接使用 from_documents 创建并写入
vector_store =  Milvus(
    embedding_function=embedding_model, # 您的 embedding 函数
    collection_name="milvus_collection",
    connection_args={
        "host": "localhost", # 指向 Docker 容器
        "port": "19530",      # 映射的端口
    },
    index_params={"index_type": "FLAT", "metric_type": "L2"},  # 索引类型和距离度量
    auto_id=True,  # 推荐：避免兼容性问题
    drop_old=True  # 首次运行可设为 True 以重建集合          
)
print(f"vector_store model--2--: {vector_store}")
# 第二步：将文档分割块写入 Milvus
# vector_store.add_documents(documents=splits)

# 1. 生成自定义ID（这里用1-10的字符串，实际可替换为业务ID）
ids = [str(i + 1) for i in range(len(splits))]
print(f"自定义文档ID列表：{ids}")

# 2. 批量插入文档（指定ids参数）
insert_result = vector_store.add_documents(
    documents=splits,
    ids=ids  # 绑定文档与ID的映射关系
)

# 3. 解析插入结果
print(f"\n插入操作返回结果：{insert_result}")
print(f"成功插入的文档ID：{insert_result}")  # 返回插入成功的ID列表

# 4. 验证插入结果（通过ID查询）
def verify_inserted(ids):
    """验证文档是否插入成功"""
    results = vector_store.get(ids=ids)  # 根据ID查询文档
    return len(results["documents"]) == len(ids)

if verify_inserted(ids):
    print("✅ 批量插入（指定ID）成功！")
else:
    print("❌ 批量插入失败！")

print(f"vector_store model--3--: {vector_store}")
#### 5. 创建检索器
retriever = vector_store.as_retriever()         # 创建检索器，用于后续从向量存储中检索相关文档


#### 6. 检索与生成部分（RETRIEVAL and GENERATION）

# 6.1 提示模板（Prompt Template）
# 定义一个模板字符串，用于构建提示，要求基于提供的上下文来回答问题
template = """Answer the question based only on the following context: 
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)  # 使用模板创建聊天提示模板

# 6.2 LLM 配置（使用 DeepSeek）
# LLM - 使用 DeepSeek，通过 ChatOpenAI 的 openai_api_base 指向 DeepSeek 接口
llm = ChatOpenAI(
    model_name=deepseek_model,                 # 指定使用的模型名称
    temperature=0.1,                           # 设置温度参数，控制输出的随机性，值越小输出越确定
    openai_api_key=api_key,                    # 设置 API 密钥
    openai_api_base=base_url                   # 设置 API 基础 URL，指向 DeepSeek
)

print(f"LLM model--3--: {llm}")

# 6.3 后处理函数（Post-processing）
# 将检索结果拼接到提示模板的 {context} 位置
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 6.4 RAG 链（Chain）
# 使用 LangChain LCEL（LangChain Expression Language）语法，通过管道符 | 串联组件：
# - retriever 检索相关文档 → format_docs 格式化为字符串 → 填入 {context}
# - RunnablePassthrough() 直接传递用户输入的问题 → 填入 {question}
# - prompt 格式化完整提示 → llm 生成回答 → StrOutputParser 解析为字符串
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(f"LLM rag_chain--4--: {rag_chain}")

# 总结
# 该代码演示了一个完整的 RAG 流程：
# 加载 PDF → 分割文档 → 生成向量并存入本地 Milvus。
# 构建提示模板，配置 LLM（通过 OpenAI 兼容接口调用 DeepSeek）。
# 使用 LCEL 将检索器、提示模板、LLM 和输出解析器串联成链。
# 最终 rag_chain 可用来回答基于文档内容的问题。

文档块数量--0--: 1
Embedding model--1--: base_url='http://localhost:11434' model='nomic-embed-text' embed_instruction='passage: ' query_instruction='query: ' mirostat=None mirostat_eta=None mirostat_tau=None num_ctx=None num_gpu=None num_thread=None repeat_last_n=None repeat_penalty=None temperature=None stop=None tfs_z=None top_k=None top_p=None show_progress=False headers=None model_kwargs=None


The ids parameter is ignored when auto_id is True. The ids will be generated automatically.


emb---- 768
vector_store model--2--: <langchain_milvus.vectorstores.milvus.Milvus object at 0x000001F1FAE0C810>
自定义文档ID列表：['1']


ConnectionNotExistException: <ConnectionNotExistException: (code=1, message=should create connection first.)>

In [ ]:
# Question
from pprint import pprint

pprint(rag_chain.invoke("What is this document about?"))